# Load the Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
import ast
import timeit
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt

repo_dir = Path.cwd().parent
data_dir = repo_dir / "data"

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

from detection import evaluate_event_detection
from detection import detect_anomalies

channel = "P-1"

train = np.load(data_dir / "train" / f"{channel}.npy")
test = np.load(data_dir / "test" / f"{channel}.npy")

# Forecast using a Rolling Average

In [2]:
def rolling_average_forecast(train,test,window):
    data=np.concatenate((train[-window:],test))
    prediction=np.mean(train[-window:])
    forecast=np.zeros(len(test))
    
    for i in range(len(test)):
        forecast[i]=prediction
        prediction+=(test[i]-data[i])/window
        
    return forecast

In [3]:
# Load the saved LSTM forecasts for alignment and later comparison
run_dir = data_dir / "2018-05-19_15.00.10"
lstm_predictions = np.load(run_dir / "y_hat" / f"{channel}.npy")

history_length = 250
n_predictions = 10

# Original test positions associated with the saved LSTM forecasts
evaluation_indices = np.arange(history_length, history_length + len(lstm_predictions))

# Actual readings we will evaluate
actual_readings = test[evaluation_indices, 0]

training_data=train[np.arange(0,2872),0]

# For each evaluated position t, predict using the observed reading at t - 1
forecast = rolling_average_forecast(training_data,actual_readings,5)

assert len(forecast) == len(lstm_predictions)

print("Evaluated readings:", len(actual_readings))
print("First test index:", evaluation_indices[0])
print("Last test index:", evaluation_indices[-1])

pd.DataFrame({
    "test_index": evaluation_indices,
    "actual": actual_readings,
    "rolling_average_forecast": forecast,
    "saved_LSTM_forecast": lstm_predictions}).head()

Evaluated readings: 8245
First test index: 250
Last test index: 8494


,test_index,actual,rolling_average_forecast,saved_LSTM_forecast
0,250,-0.540196,-0.704197,-0.454548
1,251,-0.556930,-0.673886,-0.348632
2,252,-0.227355,-0.644461,-0.082707
3,253,-0.220808,-0.544781,0.251563
4,254,0.417243,-0.451026,0.264749


In [4]:
intervals = [[2149, 2349], [3539, 3779], [4536, 4844]]

# RMSE and MAE calculations:
When we calculate the MAE and the RMSE of both the errors for the rolling average forecast and the LSTM predicitions, we see that the rolling average forecast performs significantly worse.

In [5]:
comparison = pd.DataFrame({
    "test_index": np.arange(len(actual_readings)),
    "actual": actual_readings,
    "rolling_average_forecast": forecast
})

comparison.head()

,test_index,actual,rolling_average_forecast
0,0,-0.540196,-0.704197
1,1,-0.556930,-0.673886
2,2,-0.227355,-0.644461
3,3,-0.220808,-0.544781
4,4,0.417243,-0.451026


In [6]:
mae_LSTM = np.mean(np.abs(actual_readings - lstm_predictions))
rmse_LSTM = np.sqrt(np.mean((actual_readings - lstm_predictions)**2))

mae_roll = np.mean(np.abs(actual_readings - forecast))
rmse_roll = np.sqrt(np.mean((actual_readings - forecast)**2))

print(f"LSTM MAE : {mae_LSTM:.4f}")
print(f"Rolling MAE: {mae_roll:.4f}")

print(f"LSTM RMSE : {rmse_LSTM:.4f}")
print(f"Rolling RMSE: {rmse_roll:.4f}")

LSTM MAE : 0.1701
Rolling MAE: 0.2209
LSTM RMSE : 0.2167
Rolling RMSE: 0.3155


## Now we use the detection.py module to flag anomalous intervals

In [7]:

detection_settings = {
    "history_length": 250,
    "n_predictions": 10,
    "target_column": 0,
    "batch_size": 70,
    "window_size": 30,
    "smoothing_perc": 0.05,
    "error_buffer": 100,
    "p": 0.13,
    "initial_adjustment": True,
    "initial_ignore": 500}

rolling_average_detection = detect_anomalies(test_data=test, predictions=forecast, 
                                   **detection_settings)

rolling_average_evaluation = evaluate_event_detection(detected_intervals=rolling_average_detection["detected_intervals"],
labeled_intervals=intervals)

LSTM_detection=detect_anomalies(test_data=test, predictions=lstm_predictions, 
                                   **detection_settings)

LSTM_evaluation = evaluate_event_detection(detected_intervals=LSTM_detection["detected_intervals"],
labeled_intervals=intervals)

print("Detected alert intervals:", rolling_average_detection["detected_intervals"])
print("Known anomaly intervals:", intervals)

pd.DataFrame([
    {
        "method": "Rolling average forecast based detection",
        "TP": rolling_average_evaluation["TP"],
        "FP": rolling_average_evaluation["FP"],
        "FN": rolling_average_evaluation["FN"]
    },
    {
        "method": "LSTM forecast based detection",
        "TP": LSTM_evaluation["TP"],
        "FP": LSTM_evaluation["FP"],
        "FN": LSTM_evaluation["FN"]
    }
]).set_index("method")

Detected alert intervals: [(2129, 2349)]
Known anomaly intervals: [[2149, 2349], [3539, 3779], [4536, 4844]]


,TP,FP,FN
method,,,
Rolling average forecast based detection,1,0,2
LSTM forecast based detection,3,1,0


# Then we compute the runtime and cost of the implementing the model

In [8]:
import timeit

timeit.timeit(
    lambda: rolling_average_forecast(
        training_data,
        actual_readings,
        30
    ),
    number=100
) / 100

0.016401276000542565

In [9]:
forecast_operation = lambda: rolling_average_forecast(
    training_data,
    actual_readings,
    30
)

predictions = forecast

def run_complete_pipeline():

    predictions = rolling_average_forecast(
        training_data,
        actual_readings,
        30
    )

    return detect_anomalies(
        test_data=test,
        predictions=predictions,
        **detection_settings
    )

operations = {
    "Forecast generation": forecast_operation,

    "Detection only": lambda: detect_anomalies(
        test_data=test,
        predictions=predictions,
        **detection_settings
    ),

    "Complete pipeline": run_complete_pipeline
}

runtime_rows = []

with threadpool_limits(limits=1):

    for stage, operation in operations.items():

        operation()
        print(stage)
        print(operation)

        timer = timeit.Timer(operation)

        calls_per_repeat, _ = timer.autorange()
        print(stage, calls_per_repeat)

        elapsed = np.asarray(
            timer.repeat(repeat=7,
                         number=calls_per_repeat)
        )

        milliseconds = elapsed / calls_per_repeat * 1000

        runtime_rows.append({
            "stage": stage,
            "calls per repeat": calls_per_repeat,
            "median ms": np.median(milliseconds),
            "25th percentile ms": np.percentile(milliseconds, 25),
            "75th percentile ms": np.percentile(milliseconds, 75)
        })

runtime_results = pd.DataFrame(runtime_rows)

display(runtime_results)

Forecast generation
<function <lambda> at 0x000002590B0B7EC0>
Forecast generation 20
Detection only
<function <lambda> at 0x000002590B0E4040>
Detection only 1
Complete pipeline
<function run_complete_pipeline at 0x000002590B0E4400>
Complete pipeline 1


,stage,calls per repeat,median ms,25th percentile ms,75th percentile ms
0,Forecast generation,20,17.446525,16.814085,21.023888
1,Detection only,1,556.441800,546.933200,564.667700
2,Complete pipeline,1,602.453600,572.127300,610.044450


In [10]:
%timeit forecast_operation()
forecast_operation()[:5]
forecast[:5]

24.6 ms ± 5.14 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


array([-0.7041968 , -0.67388586, -0.6444614 , -0.54478053, -0.451026  ])

In [11]:
runtime_reference = pd.DataFrame([
    ["Rolling Average", "Forecast generation", 18.06, 16.80, 18.36],
    ["Rolling Average", "Detection only", 548.18, 544.34, 621.31],
    ["Rolling Average", "Complete pipeline", 647.27, 624.90, 667.29]
], columns=[
    "method",
    "stage",
    "median_ms",
    "q25_ms",
    "q75_ms"
])

In [12]:
import numpy as np
import pandas as pd

# Runtime measurements from your benchmark
runtime_reference = pd.DataFrame([
    ["Rolling Average", "Forecast generation", 18.06, 16.80, 18.36],
    ["Rolling Average", "Detection only", 548.18, 544.34, 621.31],
    ["Rolling Average", "Complete pipeline", 647.27, 624.90, 667.29]
], columns=[
    "method",
    "stage",
    "median_ms",
    "q25_ms",
    "q75_ms"
])

# Published AWS Fargate Linux/x86 example rates
cpu_price_per_second = 0.000011244
memory_price_per_gb_second = 0.000001235

allocated_vcpus = 1
allocated_memory_gb = 2

batches_per_day = 1000
days_per_month = 30
startup_seconds_per_day = 30

runtime_multipliers = [0.5, 1.0, 2.0]

allocated_price_per_second = (
    allocated_vcpus * cpu_price_per_second
    + allocated_memory_gb * memory_price_per_gb_second
)

pipeline_times = (
    runtime_reference.loc[
        runtime_reference["stage"] == "Complete pipeline"
    ]
    .set_index("method")["median_ms"]
    / 1000
)

cost_rows = []

for multiplier in runtime_multipliers:

    for method, batch_seconds in pipeline_times.items():

        processing_seconds = (
            batches_per_day
            * batch_seconds
            * multiplier
        )

        daily_task_seconds = (
            processing_seconds
            + startup_seconds_per_day
        )

        billed_seconds_per_day = max(
            60,
            int(np.ceil(daily_task_seconds))
        )

        monthly_cost = (
            billed_seconds_per_day
            * days_per_month
            * allocated_price_per_second
        )

        cost_rows.append({
            "method": method,
            "runtime multiplier": multiplier,
            "processing seconds per day": processing_seconds,
            "billed seconds per day": billed_seconds_per_day,
            "estimated monthly USD": monthly_cost
        })

cost_estimates = pd.DataFrame(cost_rows)

display(cost_estimates.round(4))

always_running_monthly_cost = (
    days_per_month
    * 24
    * 3600
    * allocated_price_per_second
)

print(
    f"Same allocation running continuously for 30 days: "
    f"${always_running_monthly_cost:.2f}"
)

,method,runtime multiplier,processing seconds per day,billed seconds per day,estimated monthly USD
0,Rolling Average,0.5,323.635,354,0.1456
1,Rolling Average,1.0,647.270,678,0.2789
2,Rolling Average,2.0,1294.540,1325,0.5451


Same allocation running continuously for 30 days: $35.55
